# Diffusion Models
(As of 2026) The most advanced way of generating images and videos.


---


## Training
Diffusion models learn to generate images from pure noise (randomly generated pixels).

We can help them on their way by adding a little bit more noise to an image every **time step** (often referred to as $t$), so that the image gets more and more blurry as $t$ increases.

To train the models, we get them to predict how much noise is in each image, and where. Then, we calculate the difference between their prediction and the actual noise, and adjust their parameters based on that difference to improve performance.


---


## Inference
Give the model some random noise, and get it to remove the noise from the image.

In [ ]:
# DDPM by https://medium.com/data-science/diffusion-model-from-scratch-in-pytorch-ddpm-9d9760528946$0
# Corrections for the implementation of the DDPM inference formula were made by Courtney (hepialidae)

import torch
import torch.nn as nn
import torch.nn.functional as F
try:
  from einops import rearrange
except:
  !pip install einops
  from einops import rearrange
from typing import List
import random
import math
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
try:
  from timm.utils import ModelEmaV3
except:
  !pip install timm
  from timm.utils import ModelEmaV3
from tqdm import tqdm
import matplotlib.pyplot as plt
import torch.optim as optim
import numpy as np
from pathlib import Path
from datasets import load_dataset
from transformers import AutoImageProcessor, AutoModelForImageClassification

device = "cuda" if torch.cuda.is_available() else "cpu" # cuda = gpu

# Load dataset
image_size = 64
subset_size = 100000 # has to be divisible by batch_size!!!
batch_size = 16

transform = transforms.Compose([
    transforms.Resize(size=(image_size, image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

ds = load_dataset("ILSVRC/imagenet-1k", 
                  token="hugging_face_token_here", 
                  split="train", 
                  cache_dir="/kaggle/working/hf_cache",
                 streaming=True)
# ds = load_dataset("zh-plus/tiny-imagenet", token="hugging_face_token_here")

ds = ds.shuffle(buffer_size=5000, seed=42)

small_ds = ds.take(subset_size)

def transform_fn(batch):
    batch["image"] = [transform(img.convert("RGB")) for img in batch["image"]]
    return batch

small_ds = small_ds.map(transform_fn, batched=True)

def collate_fn(batch):
    images = torch.stack([item["image"] for item in batch])
    labels = torch.tensor([item["label"] for item in batch])
    return images, labels

train_dataloader = DataLoader(small_ds, batch_size=batch_size, collate_fn=collate_fn)

# Load model
image_processor = AutoImageProcessor.from_pretrained("microsoft/resnet-18")
classifier = AutoModelForImageClassification.from_pretrained("microsoft/resnet-18").to("cuda")


# Sinusoidal Embeddings
Basically the vector way of saying "the current time step is (for example) 10."

Origin explained here: https://mfaizan.github.io/2023/04/02/sines.html

---

## Formula Breakdown
Given ONE scalar time step $t$ and embedded dimensions (how long the vector is) $d$:

 SinusoidalEmbeddings(t) for one unit (e.g. image):

\begin{bmatrix}
\sin\left(t \cdot \omega_0\right), \\
\cos\left(t \cdot \omega_0\right), \\
\sin\left(t \cdot \omega_1\right), \\
\cos\left(t \cdot \omega_1\right), \\
\vdots \\
\sin\left(t \cdot \omega_{\frac{d}{2}-1}\right), \\
\cos\left(t \cdot \omega_{\frac{d}{2}-1}\right) \\
\end{bmatrix}

Where $\omega$ is the frequency (speed of oscillation - going up and down) of the sine/cosine waves.

Note that this column vector is $d$ elements long.

To calculate $\omega$ (the frequency) for each $i$ (embedding dimension):

 $\omega_i = \frac{1}{10000^{\frac{2i}{d}}}$

 An alternate version of the formula, used in PyTorch:

 $\omega_i = \exp\left( -\frac{2i}{d} \ln(10000) \right)$



In [ ]:
class SinusoidalEmbeddings(nn.Module):
    def __init__(self, time_steps : int, # maximum number of time steps
                 embed_dim : int): # the length of the embedding vector. must be even and twice
        super().__init__()

        # map out time steps (t)
        position = torch.arange(time_steps).unsqueeze(1).float()

        # the frequency scaling factor for each feature channel pair (1 / 10000^(2i/d))
        div = torch.exp(torch.arange(0, embed_dim, step=2).float() * -(math.log(10000.0) / embed_dim))

        # instantiate the embeddings tensor (shape = [time_steps, embed_dim])
        embeddings = torch.zeros(time_steps, embed_dim, requires_grad=False).to(device)

        # sin and cos in alternating columns
        embeddings[:, 0::2] = torch.sin(position * div) # representation of PE(t, 2i) = sin(t / 10000^(2i/d))
        embeddings[:, 1::2] = torch.cos(position * div) # representation of PE(t, 2i + 1) = cos(t / 10000^(2i/d))
        self.embeddings = embeddings

    def forward(self, x, # the input (e.g. feature map)
                t): # current time step index
        embeds = self.embeddings[t].to(x.device)
        return embeds[:, :, None, None]

In [ ]:
## Residual Blocks

class ResBlock(nn.Module):
    def __init__(self, C: int, # amount of channels used by Conv2D (must be divisible by num_groups)
                 num_groups: int, # number of groups to separate the channels into in nn.GroupNorm
                 dropout_prob: float): # the probability that an element will be changed to 0 in self.dropout
        super().__init__()
        self.relu = nn.ReLU(inplace=True)
        self.gnorm1 = nn.GroupNorm(num_groups=num_groups, num_channels=C)
        self.gnorm2 = nn.GroupNorm(num_groups=num_groups, num_channels=C)
        self.conv1 = nn.Conv2d(C, C, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(C, C, kernel_size=3, padding=1)
        self.dropout = nn.Dropout(p=dropout_prob, inplace=True) # randomly changes some elements to 0

    def forward(self, x, # input
                embeddings): # the embeddings of the input
        # embeddings are necessary here because it provides time step info!
        x = x + embeddings[:, :x.shape[1], :, :] # slices embeddings to fit x's channel count
        r = self.conv1(self.relu(self.gnorm1(x)))
        r = self.dropout(r)
        r = self.conv2(self.relu(self.gnorm2(r)))
        return r + x

In [ ]:
class Attention(nn.Module):
    def __init__(self, C : int, # number of channels in nn.Linear layers
                 num_heads : int, # the number of heads to use in F.scaled_dot_product_attention (H)
                 dropout_prob : float): # probability of dropout in F.scaled_dot_product_attention (can be 0)
        super().__init__()
        self.proj1 = nn.Linear(C, C*3) # splits input into 3 for Q, K, and V values
        self.proj2 = nn.Linear(C, C) # mixes the heads' outputs together and allows the model to learn to fit the standard feature mappings
        self.num_heads = num_heads
        self.dropout_prob = dropout_prob

    def forward(self, x):
        h, w = x.shape[2:]
        x = rearrange(x, 'b c h w -> b (h w) c') # [b, c, h, w] -> [b, h*w, c]
        x = self.proj1(x) # [b, L, C] -> [b, L, 3C]
        x = rearrange(x, 'b L (C H K) -> K b H L C', K=3, H=self.num_heads) # [b, L, 3C] -> [b, L, C*H*K] -> [K, b, H, L, C]
        q,k,v = x[0], x[1], x[2] # [K, b, H, L, C] -> [b, H, L, C] * 3
        x = F.scaled_dot_product_attention(q,k,v, is_causal=False, dropout_p=self.dropout_prob) # [b, H, L, C] * 3 -> [b, H, L, C] where the magic happens!
        x = rearrange(x, 'b H (h w) C -> b h w (C H)', h=h, w=w) # [b, H, L, C] -> [b, H, h*w, C] -> [b, h, w, C*H] -> [b, h, w, C]
        x = self.proj2(x) # [b, h, w, C] -> [b, h, w, C]
        return rearrange(x, 'b h w C -> b C h w')

In [ ]:
class UnetLayer(nn.Module):
    def __init__(self,
                upscale: bool, # whether to upscale the image
                attention: bool, # whether to use attention
                num_groups: int, # amount of groups for the two Residual Network blocks
                dropout_prob: float, # dropout probability for the two Residual Network blocks and the attention layer (if used)
                num_heads: int, # amount of heads for the attention layer (if used)
                C: int): # amount of channels for the two Residual Network blocks
        super().__init__()

        self.ResBlock1 = ResBlock(C=C, num_groups=num_groups, dropout_prob=dropout_prob)
        self.ResBlock2 = ResBlock(C=C, num_groups=num_groups, dropout_prob=dropout_prob)

        # encoder = no upscale; decoder = upscale
        if upscale:
            self.conv = nn.ConvTranspose2d(C, C//2, kernel_size=4, stride=2, padding=1)
        else:
            self.conv = nn.Conv2d(C, C*2, kernel_size=3, stride=2, padding=1)

        if attention:
            self.attention_layer = Attention(C, num_heads=num_heads, dropout_prob=dropout_prob)

    def forward(self, x, embeddings):
        x = self.ResBlock1(x, embeddings)
        if hasattr(self, 'attention_layer'): # check if self.attention_layer exists
            x = self.attention_layer(x)
        x = self.ResBlock2(x, embeddings)
        return self.conv(x), x

class UNET(nn.Module):
    def __init__(self,
                Channels: List = [64, 128, 256, 512, 512, 384], # list of channels for the U-Net layers.
                # (above) This isn't shaped like a U-Net, leading me to believe this is a bug. However, people have ran the code and said it worked.
                Attentions: List = [False, True, False, False, False, True], # whether to use attention for each of the layers
                Upscales: List = [False, False, False, True, True, True], # whether to upscale the image for each of the layers
                num_groups: int = 16, # number of groups for ResBlock's nn.GroupNorm
                dropout_prob: float = 0.1,
                num_heads: int = 8,
                input_channels: int = 3, # how many channels the input has
                output_channels: int = 3, # how many channels you want the output to have
                time_steps: int = 1000): # maximum number of time steps
        super().__init__()
        self.num_layers = len(Channels)
        self.shallow_conv = nn.Conv2d(input_channels, Channels[0], kernel_size=3, padding=1)
        out_channels = (Channels[-1]//2)+Channels[0]
        self.late_conv = nn.Conv2d(out_channels, out_channels//2, kernel_size=3, padding=1)
        self.output_conv = nn.Conv2d(out_channels//2, output_channels, kernel_size=1)
        self.relu = nn.ReLU(inplace=True)
        self.embeddings = SinusoidalEmbeddings(time_steps=time_steps, embed_dim=max(Channels))

        # Define the different layers of the U-Net, and store them inside the class.
        for i in range(self.num_layers):
            layer = UnetLayer(
                upscale=Upscales[i],
                attention=Attentions[i],
                num_groups=num_groups,
                dropout_prob=dropout_prob,
                C=Channels[i],
                num_heads=num_heads
            )
            setattr(self, f'Layer{i+1}', layer)

    def forward(self, x, # input
                t): # time step
        x = self.shallow_conv(x)

        residuals = []

        # Encoding
        for i in range(self.num_layers//2):
            layer = getattr(self, f'Layer{i+1}') # access each layer of the U-Net
            embeddings = self.embeddings(x, t)
            x, r = layer(x, embeddings)
            residuals.append(r)

        # Decoding
        for i in range(self.num_layers//2, self.num_layers):
            layer = getattr(self, f'Layer{i+1}')

            # string together the outputs and residuals at dim=1
            x = torch.concat((layer(x, embeddings)[0], residuals[self.num_layers-i-1]), dim=1)

        return self.output_conv(self.relu(self.late_conv(x))) # final output edits

In [ ]:
# basically: when given time, how much should I scale the noise?

class DDPM_Scheduler(nn.Module):
    def __init__(self, num_time_steps: int=1000): # maximum number of time steps
        super().__init__()

        # creates a vector of size num_time_steps, where elements start at 1e-4 and gradually increase (equally!) until 0.02
        self.beta = torch.linspace(1e-4, 0.02, num_time_steps, requires_grad=False) # no gradients since it's a constant

        self.alpha = 1 - self.beta
        self.alpha.requires_grad_(False)
        self.alpha_cumprod = torch.cumprod(self.alpha, dim=0).requires_grad_(False) # cumulative products of elements in dim; basically = [0, 0*1, 0*1*2, ...]

    def forward(self, t):
        return self.beta[t], self.alpha[t], self.alpha_cumprod[t]

In [ ]:
def set_seed(seed: int = 42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # relating to cuDNNs (cuda Deep Neural Networks, which make DNNs run faster on GPU)
    torch.backends.cudnn.deterministic = True # only use deterministic (same output every time with same inputs) convolution algorithms!!
    torch.backends.cudnn.benchmark = False # if true, benchmark multiple convolution algorithms and select the fastest

    np.random.seed(seed)
    random.seed(seed)

# Exponential Moving Average (EMA)
I found this website very helpful: https://towardsdatascience.com/intuitive-explanation-of-exponential-moving-average-2eb9693ea4dc/$0.
It also explains bias correction, a core part of EMA.

In machine learning, EMA keeps track of trends in a model's parameters, smoothing the parameter changing trend. This allows for better generalization.

The formula for EMA at a time $\text{t}$ is $\text{EMA}_t = \beta \cdot \text{EMA}_{t-1} + (1 - \beta) \cdot \theta_t$, where:

* $\theta_t$ are the model's current parameters;
* $\beta$ is a hyperparameter between 0 and 1 that defines how important the past vs present is. This is usually 0.9999 or 0.99999, but theoretically, it could be anything;
* $\text{EMA}_{t-1}$ is the EMA for the previous iteration/epoch.

# Step-by-Step Forward Diffusion
For some more math: https://medium.com/@steinsfu/diffusion-model-clearly-explained-cd331bd41166$0

To start off, let's describe our problem.

We need a way to gradually add Gaussian noise to an image $x_0$ over discrete (countable) time-steps $T$, until it becomes the noisy image $x_T$. We can express this problem as:

$q(x_1, x_2, \dots, x_T \mid x_0) = \prod_{t=1}^{T} q(x_t \mid x_{t-1})$

In readable English, this means "everything between $x_1$ and $x_T$, made from $x_0$, equals the product of all the transformations."

Let's zoom in and see how one transformation works.

$q(x_t \mid x_{t-1}) = \mathcal{N}\left(x_t; \sqrt{1 - \beta_t} \, x_{t-1}, \beta_t \mathbf{I}\right)$

Where:
*   $\mathcal{N}$ is the Gaussian noise function;
*   $\beta_t$ is the noise scheduler (scales how much noise is in an image) at time $t$;
*   $\mathbf{I}$ is an identity matrix matching the dimensionality of $x_0$.

Note that $\sqrt{1 - \beta_t} \, x_{t-1}$ is referred to as the mean $\mu_t$, and $\beta_t\mathbf{I}$ is referred to as the variance $\Sigma_t$.

If we apply the reparameterization trick iteratively, then expand the formula recursively, we'll be able to find $x_t$ at any time-step without needing to go through the time-steps in between! Let's see how the formula changes:

$x_t = \sqrt{\bar{\alpha}_t} \, x_0 + \sqrt{1 - \bar{\alpha}_t} \, \epsilon, \quad \text{where } \epsilon \sim \mathcal{N}(0, \mathbf{I})$

Where:
*   $\bar\alpha_t$ is the cumulative signal factor (how much clean data was retained) from time-step 0 to $t$.

...And we're done!

In [ ]:
def train(batch_size: int=16,
          num_time_steps: int=1000,
          num_epochs: int=15,
          seed: int=-1,
          ema_decay: float=0.9999,
          lr=2e-5,
          checkpoint_path: Path=None,
          load_checkpoint : bool = True):
    # if a seed is not specified, randomize one. else, use the set_seed() function.
    set_seed(random.randint(0, 2**32-1)) if seed == -1 else set_seed(seed)

    # initialize scheduler, model, loss function, optimizer, exponential moving average (EMA), and model save folders
    scheduler = DDPM_Scheduler(num_time_steps=num_time_steps)
    model = UNET().cuda()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    ema = ModelEmaV3(model, decay=ema_decay)
    if load_checkpoint == True and (checkpoint_path / "model.pth").exists() == True:
      model.load_state_dict(torch.load(f=checkpoint_path/"model.pth"))
      ema.load_state_dict(torch.load(f=checkpoint_path/"ema.pth"))
      optimizer.load_state_dict(torch.load(f=checkpoint_path/"optimizer.pth"))
    else:
      checkpoint_path.mkdir(parents=True, exist_ok=True)

      criterion = nn.MSELoss(reduction='mean')

      for i in range(num_epochs):
        total_loss = 0

        # bidx = batch index; x = image batch
        for bidx, (x, _) in enumerate(tqdm(train_dataloader, desc=f"Epoch {i+1}/{num_epochs} | Max: {subset_size // batch_size} iterations")):
            x = x.cuda()

            # creates a tensor the length of batch_size, filled with random integers from 0-num_time_steps
            # this is used to determine how much noise is in each of the image, with higher time steps having more noise
            t = torch.randint(0,num_time_steps,(batch_size,))

            # create the noise that is added to the image
            e = torch.randn_like(x, requires_grad=False)

            # a is used to scale the noise (e) using the current time step
            a = scheduler.alpha_cumprod[t].view(batch_size,1,1,1).cuda()

            print(x.shape)

            # the forward diffusion equation
            x = (torch.sqrt(a)*x) + (torch.sqrt(1-a)*e)

            output = model(x, t)
            optimizer.zero_grad()

            # the loss is calculated against e because the model is supposed to predict what noise is in the image
            loss = criterion(output, e)

            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            ema.update(model)

        print(f'Epoch {i+1} | Loss {total_loss/(subset_size/batch_size):.5f}') # 60000 is the size of the MNIST training dataset

      # save stuff
      torch.save(obj=model.state_dict(), f=checkpoint_path / "model.pth")
      torch.save(obj=optimizer.state_dict(), f=checkpoint_path / "optimizer.pth")
      torch.save(obj=ema.state_dict(), f=checkpoint_path / "ema.pth")

# Reverse Diffusion
Now that we can go from $x_{t-1}$ to $x_t$, would it be possible to go backwards? We can't do that using $q(\cdot)$, but we could train a neural network $p_\theta$ to do it.

The formula for one step of reverse diffusion from *Denoising Diffusion Probabilistic Models* by Ho et al is:

$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right) + \sigma_t z, \quad \text{where } z \sim \mathcal{N}(0, \mathbf{I})$

Where:
*   $\alpha_t$ is the retention factor (how much clean data was preserved) from $x_{t-1}$ to $x_t$;
*   $\epsilon$ is the neural network that takes in $x_t$ and $t$, then outputs its prediction of the noise in an image;
*   $\sigma_t$ is the standard deviation of the reverse step noise. It is set to $\sqrt{\beta_t}$ in this implementation.

In [ ]:
def unnormalize_imagenet(tensor: torch.Tensor) -> torch.Tensor:
    mean = torch.tensor([0.485, 0.456, 0.406]).view(-1, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(-1, 1, 1)

    # Reverse: x = (normalized * std) + mean
    unnormalized = tensor * std + mean
    return unnormalized

def display_reverse(images: List):
    fig, axes = plt.subplots(1, 10, figsize=(10,1)) # would be a better function if we could customize this
    for i, ax in enumerate(axes.flat):
        x = unnormalize_imagenet(images[i]).squeeze(0)
        x = rearrange(x, 'c h w -> h w c')
        x = x.detach().numpy()
        ax.imshow(x)
        ax.axis('off')
    plt.show()

def inference(checkpoint_path: Path=None,
              num_time_steps: int=1000,
              ema_decay: float=0.9999,
              adv_scale : float = 0.5, # adversarial guidance scale for AdvDiff (how much we should value the adversarial guidance)
              target : torch.Tensor | None = None, # target class for images (assuming one image)
              adv : bool = False): # whether adversarial guidance should be implemented
    torch.cuda.empty_cache()
    # load a bunch of stuff
    model = UNET().cuda()
    model.load_state_dict(torch.load(f=checkpoint_path/"model.pth"))
    ema = ModelEmaV3(model, decay=ema_decay)
    ema.load_state_dict(torch.load(f=checkpoint_path/"ema.pth"))
    scheduler = DDPM_Scheduler(num_time_steps=num_time_steps)
    times = [0,15,50,100,200,300,400,550,700,999] # this is later reversed to show the process of making the image
    images = [] # where the finished images go
    if target == None:
        target = torch.randint(size=[1], low=0, high=1000).cuda()
    else:
        target = target.cuda()

    model = ema.module.eval()

    z = torch.randn(1, 3, 64, 64) # randomly generate an RGB image
    for t in tqdm(reversed(range(1, num_time_steps))):
        with torch.no_grad():
            t = [t]

            # reverse diffusion, note that z is a stand-in for x_t
            noise_pred = (scheduler.beta[t[0]] / (torch.sqrt(1 - scheduler.alpha_cumprod[t[0]]))) * (model(z.cuda(), t)).cpu() # predicted noise + coefficient
            coef = (1 / torch.sqrt(scheduler.alpha[t[0]]))
            z, noise_pred = z * coef, noise_pred * coef
            # z1 = z.clone()
            z = z - noise_pred
            # z2 = z.clone()

            if t[0] in times:
                images.append(z.clone())
                # print(f"{t[0]} z1: {z1}")
                # print(f"{t[0]} z2: {z2}")

            # add a little random noise to the image (enhances image quality)
            e = torch.randn(1, 3, 64, 64)
            sigma = torch.sqrt(scheduler.beta[t[0]])
            z = z + (e * sigma) # sigma_t*z here
            # z3 = z.clone()

            # if t[0] in times:
                # print(f"{t[0]} z3: {z3}")

        if adv == True:
            # add adversarial guidance
            new_z = z.requires_grad_(True)
            
            logits = classifier(new_z.to("cuda")).logits
    
            # calculate log likelihood
            log_probs = F.log_softmax(logits, dim=-1)
            target_log_prob = log_probs[torch.arange(z.shape[0]), target].sum()
    
            # get the gradient
            grad = torch.autograd.grad(outputs=target_log_prob, inputs=new_z)[0].to("cpu") # find gradient of log likelihood
    
            with torch.no_grad():
              new_z += (sigma ** 2) * adv_scale * grad # final output
              z = new_z

    # plot final image
    noise_pred = (scheduler.beta[0] / (torch.sqrt(1 - scheduler.alpha_cumprod[0]))) * ((model(z.cuda(), torch.tensor([0]).cuda()).cpu())) # predicted noise + coefficient
    coef = (1 / torch.sqrt(scheduler.alpha[0]))
    x, noise_pred = z * coef, noise_pred * coef
    x = x - noise_pred
    
    images.append(x)
    new_x = rearrange(unnormalize_imagenet(x).squeeze(0), 'c h w -> h w c').detach()
    new_x = new_x.numpy()
    plt.imshow(new_x)
    plt.show()

    # plot everything
    display_reverse(images)

    return images[-1], target

In [ ]:
path = Path('/kaggle/input/models/hepialidae/imagenet-diffuser/pytorch/80epochs/1/imagenet_checkpoint80')
adv = True

# train(checkpoint_path=path, lr=2e-5, num_epochs=1, load_checkpoint=True)
img, target = inference(path, num_time_steps=1000)

with torch.no_grad():
    logits = classifier(img.cuda()).logits

probs = F.softmax(logits, dim=-1)
predicted_label = logits.argmax(-1).item()
conf = probs[0, predicted_label].item()
if adv:
    target_conf = probs[0, target.item()].item()

print(f"Prediction: {classifier.config.id2label[predicted_label]} | Confidence: {conf * 100:.2f}%")
if adv:
    print(f"Target: {classifier.config.id2label[target.item()]} | Target Confidence: {target_conf * 100:.2f}%")